In [ ]:
import sys
from pathlib import Path
import torch
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    TrainingArguments,
    DataCollatorWithPadding
)
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))


from project_paths import get_paths
from distillation import Phase2DistillationTrainer 
from teacher_finetune_headtail import (
    build_teacher_model, compute_metrics, calculate_class_weights
)

paths = get_paths(ROOT)
DATA_DIR = paths.data_processed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TEACHER_PATH = str(paths.checkpoints / "bert_teacher_finetuned" / "checkpoint-18000")
STUDENT_TD1_PATH = str(paths.checkpoints / "tinybert_phase1" / "student_base_final")

teacher = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_PATH, attn_implementation="eager"
).to(device)

student = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_TD1_PATH, num_labels=2
).to(device)

In [3]:
for name, param in student.named_parameters():
    assert param.requires_grad, f"FROZEN: {name}"

In [2]:
from datasets import load_dataset
print("Loading from Parquet files...")
train_ds = load_dataset("parquet", data_files=str(DATA_DIR / "train.parquet"))["train"]
val_ds = load_dataset("parquet", data_files=str(DATA_DIR / "val.parquet"))["train"]
test_ds = load_dataset("parquet", data_files=str(DATA_DIR / "test.parquet"))["train"]

print(f"Train size: {len(train_ds)}")
print(f"Val size:   {len(val_ds)}")
print(f"Test size:  {len(test_ds)}")

Loading from Parquet files...
Train size: 231423
Val size:   34912
Test size:  31344


In [ ]:
training_args = TrainingArguments(
    output_dir=str(paths.checkpoints / "tinybert_phase2"),
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,   
    num_train_epochs=3,
    learning_rate=2e-5,
    fp16=True,                        
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=5,
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    greater_is_better=True,
    report_to="none",
    dataloader_num_workers=0,
)

In [6]:
class_weights = calculate_class_weights(train_ds)
print(class_weights)

tensor([1.0000, 2.8518])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="bert-base-uncased", use_fast=True)
collator = DataCollatorWithPadding(tokenizer)

trainer = Phase2DistillationTrainer(
    teacher_model=teacher,
    model=student,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,   
    class_weights=class_weights,
    temperature=1.0,    
    rho_ok=0.9,
    rho_bad=0.2,
)
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc,Pr Auc
1000,0.380900,0.424804,0.767473,0.582794,0.545507,0.625552,0.802772,0.622437
2000,0.374100,0.406734,0.737426,0.583904,0.496028,0.709620,0.802973,0.630720
3000,0.334100,0.451053,0.758851,0.584267,0.528828,0.652692,0.802728,0.624601
4000,0.376500,0.404150,0.772285,0.591302,0.553620,0.634488,0.807186,0.634958
5000,0.350400,0.408624,0.688016,0.566298,0.443053,0.784532,0.803758,0.632661
6000,0.300000,0.414865,0.777641,0.590883,0.565634,0.618491,0.807922,0.637985
7000,0.354000,0.420123,0.774576,0.589334,0.559109,0.623014,0.807564,0.635431
8000,0.338000,0.422747,0.762660,0.586651,0.535422,0.648720,0.806496,0.636612
9000,0.335100,0.424964,0.784802,0.583790,0.586291,0.581311,0.808014,0.639814
10000,0.293500,0.428556,0.792736,0.568515,0.618609,0.525927,0.803743,0.636551


TrainOutput(global_step=21696, training_loss=0.3291674864898741, metrics={'train_runtime': 22215.1457, 'train_samples_per_second': 31.252, 'train_steps_per_second': 0.977, 'total_flos': 9900352773362688.0, 'train_loss': 0.3291674864898741})

In [ ]:
final_student_path = str(paths.checkpoints / "tinybert_phase2" / "student_final")

print(f"Salvataggio del modello student  in: {final_student_path}")
student.save_pretrained(str(final_student_path))
tokenizer.save_pretrained(str(final_student_path))
print("Salvataggio completato")

Salvataggio del modello student  in: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\checkpoints\tinybert_phase2\student_final
Salvataggio completato


: 

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    accuracy_score
)



In [4]:
def ensure_labels_column(ds):
    if "labels" in ds.column_names:
        return ds
    if "label" in ds.column_names:
        return ds.rename_column("label", "labels")
    raise ValueError("Dataset must contain 'label' or 'labels' column.")

val_ds = ensure_labels_column(val_ds)
test_ds = ensure_labels_column(test_ds)

In [5]:
STUDENT_PHASE2_PATH = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(str(STUDENT_PHASE2_PATH), use_fast=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

student = AutoModelForSequenceClassification.from_pretrained(str(STUDENT_PHASE2_PATH)).to(device)
student.eval()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
from transformers import Trainer

infer_args = TrainingArguments(
    per_device_eval_batch_size=32,   
    dataloader_num_workers=2,
    report_to="none",
    fp16=False,                      
)

infer_trainer = Trainer(
    model=student,
    args=infer_args,
    data_collator=collator,
    tokenizer=tokenizer,  
)

C:\Users\cola0\AppData\Local\Temp\ipykernel_4340\1431322795.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  infer_trainer = Trainer(


In [ ]:
def predict_probs_labels(trainer, ds):
    pred_out = trainer.predict(ds)
    logits = torch.tensor(pred_out.predictions)   
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[:, 1]
    labels = pred_out.label_ids.astype(int)
    return probs, labels

In [8]:
import numpy as np

def find_best_threshold_f1(y_true, probs, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    best = {
        "threshold": 0.5,
        "f1": -1.0,
        "precision": None,
        "recall": None,
        "accuracy": None,
    }

    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_true, preds, pos_label=1, zero_division=0)

        if f1 > best["f1"]:
            best["threshold"] = float(t)
            best["f1"] = float(f1)
            best["precision"] = float(precision_score(y_true, preds, pos_label=1, zero_division=0))
            best["recall"] = float(recall_score(y_true, preds, pos_label=1, zero_division=0))
            best["accuracy"] = float(accuracy_score(y_true, preds))

    return best

def evaluate_with_threshold(y_true, probs, threshold):
    y_pred = (probs >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) > 1 else 0.5,
        "pr_auc": float(average_precision_score(y_true, probs)) if len(np.unique(y_true)) > 1 else 0.0,
    }

In [9]:
val_probs, val_labels = predict_probs_labels(infer_trainer, val_ds)
best = find_best_threshold_f1(val_labels, val_probs)

print("Best threshold on validation:", best)

test_probs, test_labels = predict_probs_labels(infer_trainer, test_ds)
test_metrics = evaluate_with_threshold(test_labels, test_probs, best["threshold"])

print(f"\n--- TEST METRICS (Threshold: {best['threshold']:.4f}) ---")
for k, v in test_metrics.items():
    if k == "threshold":
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v:.6f}")

Best threshold on validation: {'threshold': 0.42000000000000004, 'f1': 0.4438975354431696, 'precision': 0.3236290118776851, 'recall': 0.7064210061782877, 'accuracy': 0.5404731897341888}



--- TEST METRICS (Threshold: 0.4200) ---
threshold: 0.4200
accuracy: 0.526927
f1: 0.439522
precision: 0.317375
recall: 0.714514
roc_auc: 0.623203
pr_auc: 0.353494
